# 04 - Mushroom classification met CatBoost

**Auteur:** Brent Van Gelder  
**Dataset:** `mushroom_project_dataset.csv`  
**Doel:** voorspellen of een paddenstoel eetbaar (`e`) of giftig/onbekend (`p`) is.

> Dit is een onderwijsmodel en mag nooit worden gebruikt om echte paddenstoelen veilig te verklaren.

Deze notebook gebruikt dezelfde datacontroles, uitgesloten ruiskolommen, random seed, 70/15/15-split en evaluatiemetrics als de Random Forest-notebook. CatBoost behoudt categorische waarden als categorieën; de split, kolomkeuze, ontbrekende-waardenstrategie en metrics blijven gelijk.

## Technologie

CatBoost is gradient boosting met ingebouwde ondersteuning voor categorische kenmerken. Daardoor is one-hot encoding niet nodig. Dit is een model-specifiek technisch verschil; de databron, rijen, verwijderde ruiskolommen en ontbrekende-waardenregels blijven gelijk.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier

## 1. Data en vaste instellingen

De meegeleverde project-CSV wordt lokaal geladen. De twee expliciet als `jumbled_noise` benoemde kolommen worden in alle modelnotebooks uitgesloten. We controleren labels, ontbrekende waarden en duplicaten voordat het model wordt gebouwd.

In [ ]:
RANDOM_STATE = 42
DATA_PATH = Path("mushroom_project_dataset.csv")
TARGET = "class"
EDIBLE_LABEL = "e"
POISONOUS_LABEL = "p"
RUN_FINAL_TEST = False

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset niet gevonden: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
if TARGET not in df.columns:
    raise ValueError(f"Doelkolom '{TARGET}' ontbreekt.")
if df[TARGET].isna().any():
    raise ValueError("De doelkolom bevat ontbrekende waarden.")
if set(df[TARGET].unique()) != {EDIBLE_LABEL, POISONOUS_LABEL}:
    raise ValueError(f"Onverwachte labels: {set(df[TARGET].unique())}")

noise_columns = [column for column in df.columns if column.startswith("jumbled_noise_")]
model_df = df.drop(columns=noise_columns).copy()

summary = pd.DataFrame({
    "datatype": model_df.dtypes.astype(str),
    "ontbrekend": model_df.isna().sum(),
    "ontbrekend_percentage": (100 * model_df.isna().mean()).round(1),
    "unieke_waarden": model_df.nunique(dropna=True),
}).sort_values("ontbrekend_percentage", ascending=False)

print(f"Rijen: {len(model_df):,}")
print(f"Exact dubbele rijen: {model_df.duplicated().sum()}")
print(f"Uitgesloten ruiskolommen: {noise_columns}")
display(model_df[TARGET].value_counts().rename_axis("klasse").to_frame("aantal"))
display(summary)

## 2. Dezelfde train-, validatie- en testset

Met `random_state=42` en dezelfde tweestapssplit krijgt iedere notebook exact dezelfde 70% trainingsrijen, 15% validatierijen en 15% testrijen. `stratify` bewaart de klasseverhouding. De testset blijft vergrendeld totdat alle keuzes zijn gemaakt.

In [ ]:
X = model_df.drop(columns=TARGET)
y = model_df[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "rijen": [len(X_train), len(X_validation), len(X_test)],
    "aandeel_dataset": [len(X_train) / len(X), len(X_validation) / len(X), len(X_test) / len(X)],
    "aandeel_giftig": [
        (y_train == POISONOUS_LABEL).mean(),
        (y_validation == POISONOUS_LABEL).mean(),
        (y_test == POISONOUS_LABEL).mean(),
    ],
}, index=["train", "validatie", "test"])
display(split_summary.style.format({
    "aandeel_dataset": "{:.1%}",
    "aandeel_giftig": "{:.1%}",
}))

## 3. Dezelfde cleaning met native categorieën

Numerieke ontbrekende waarden krijgen opnieuw de mediaan die uitsluitend uit de trainingsset komt. Categorische ontbrekende waarden worden opnieuw `__missing__`. CatBoost ontvangt daarna de kolomnamen van de categorische kenmerken en codeert ze intern.

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()
numeric_medians = X_train[numeric_features].median()

def clean_for_catboost(frame):
    cleaned = frame.copy()
    cleaned[numeric_features] = cleaned[numeric_features].fillna(numeric_medians)
    cleaned[categorical_features] = (
        cleaned[categorical_features]
        .fillna("__missing__")
        .astype(str)
    )
    return cleaned

X_train_clean = clean_for_catboost(X_train)
X_validation_clean = clean_for_catboost(X_validation)
X_test_clean = clean_for_catboost(X_test)

model = CatBoostClassifier(
    iterations=600,
    depth=8,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
)

model.fit(
    X_train_clean,
    y_train,
    cat_features=categorical_features,
    verbose=False,
)
print(f"CatBoost getraind met {model.tree_count_} boosting-iteraties.")

## 4. Dezelfde evaluatiemetrics

De giftige klasse `p` is de positieve klasse. F2 weegt recall zwaarder dan precision, omdat een giftige paddenstoel ten onrechte als eetbaar classificeren de ernstigste fout is. We rapporteren daarnaast accuracy, balanced accuracy, F1, ROC-AUC en average precision.

In [ ]:
def evaluate_classifier(fitted_model, X_eval, y_eval, split_name):
    predictions = fitted_model.predict(X_eval)
    poisonous_index = list(fitted_model.classes_).index(POISONOUS_LABEL)
    poisonous_probability = fitted_model.predict_proba(X_eval)[:, poisonous_index]
    y_binary = (y_eval == POISONOUS_LABEL).astype(int)

    metrics = pd.Series({
        "accuracy": accuracy_score(y_eval, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_eval, predictions),
        "precision_giftig": precision_score(
            y_eval, predictions, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "recall_giftig": recall_score(
            y_eval, predictions, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "f1_giftig": f1_score(
            y_eval, predictions, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "f2_giftig": fbeta_score(
            y_eval, predictions, beta=2, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "roc_auc": roc_auc_score(y_binary, poisonous_probability),
        "average_precision": average_precision_score(y_binary, poisonous_probability),
    }, name=split_name)

    display(metrics.to_frame().T.style.format("{:.3f}"))
    print(classification_report(
        y_eval,
        predictions,
        labels=[EDIBLE_LABEL, POISONOUS_LABEL],
        target_names=["eetbaar", "giftig/onbekend"],
        zero_division=0,
    ))
    ConfusionMatrixDisplay.from_predictions(
        y_eval,
        predictions,
        labels=[EDIBLE_LABEL, POISONOUS_LABEL],
        display_labels=["eetbaar", "giftig/onbekend"],
        cmap="Blues",
    )
    plt.title(f"Confusion matrix - {split_name}")
    plt.grid(False)
    plt.show()
    return metrics

## 5. Validatie

De validatieset wordt uitsluitend gebruikt voor de gerapporteerde metrics. Het model ziet deze rijen niet tijdens training, zodat de vergelijking met de andere modellen eerlijk blijft.

In [ ]:
validation_metrics = evaluate_classifier(
    model, X_validation_clean, y_validation, "catboost_validatie"
)

## 6. Feature importance

CatBoost rapporteert importance per oorspronkelijke kolom. Deze waarden beschrijven hoeveel het getrainde model een kenmerk gebruikt; ze bewijzen geen oorzaak-gevolgrelatie.

In [ ]:
feature_importance = (
    pd.DataFrame({
        "feature": X_train_clean.columns,
        "importance": model.get_feature_importance(),
    })
    .sort_values("importance", ascending=False)
)
display(feature_importance)
plt.figure(figsize=(8, 5))
plt.barh(
    feature_importance["feature"][::-1],
    feature_importance["importance"][::-1],
    color="royalblue",
)
plt.title("CatBoost feature importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 7. Definitieve testset - voorlopig vergrendeld

Zet `RUN_FINAL_TEST` pas op `True` nadat tuning en eventuele drempelkeuzes definitief zijn vastgelegd.

In [ ]:
if RUN_FINAL_TEST:
    test_metrics = evaluate_classifier(model, X_test_clean, y_test, "catboost_test")
else:
    print("Testset niet geëvalueerd: leg eerst alle modelkeuzes vast.")

## 8. Volgende beslissing

Mogelijke vervolgstappen zijn tuning van boomdiepte, learning rate, regularisatie en class weights. Deze keuzes worden pas na overleg toegevoegd.